<a href="https://colab.research.google.com/github/mahdisahad/WOA-Feature-Selection/blob/main/notebooks/06_Results_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="text-align: right">

# نوت‌بوک ۶: ارزیابی نهایی و تحلیل نتایج

این نوت‌بوک ویژگی‌هایی را که الگوریتم نهنگ در نوت‌بوک ۰۵ انتخاب کرد، روی **داده‌ی test** ارزیابی می‌کند و با مدل «همه‌ی ویژگی‌ها» مقایسه می‌کند.

**طراحی مقایسه:**
- دو مدل کاملاً یکسان (استانداردسازی + Logistic Regression با همان تنظیمات) آموزش می‌بینند؛ تنها تفاوت **مجموعه‌ی ویژگی‌ها** است: یکی همه‌ی ویژگی‌ها، دیگری فقط ویژگی‌های انتخاب‌شده‌ی WOA.
- هر دو فقط روی train آموزش می‌بینند و روی test (که در انتخاب ویژگی نقشی نداشت) ارزیابی می‌شوند، پس مقایسه منصفانه است.
- معیارها: Accuracy، Balanced Accuracy، Precision، Recall و F1 (Macro).

**نکته‌ها برای تفسیر:**
- test نسبتاً کوچک است؛ اختلاف یکی دو نمونه در accuracy ممکن است صرفاً نوسان تصادفی باشد.
- مدل پایه در این نوت‌بوک Logistic Regression است. مدل‌های دیگر (مثل SVM) در نوت‌بوک ۰۳ ارزیابی شده‌اند و ممکن است روی همه‌ی ویژگی‌ها بهتر باشند.
- هدف انتخاب ویژگی معمولاً **کوچک‌تر کردن مدل با حفظ کیفیت** است، نه لزوماً بالا بردن دقت.

</div>

<div dir="rtl" style="text-align: right">

### اتصال به Google Drive
Drive وصل می‌شود تا نتایج نوت‌بوک ۰۵ و دیتاست‌ها در دسترس باشند.

</div>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

<div dir="rtl" style="text-align: right">

### وارد کردن کتابخانه‌ها
- `numpy`، `pandas` و `matplotlib` برای محاسبه، جدول و نمودار.
- `Pipeline`، `StandardScaler` و `LogisticRegression` برای ساخت دو مدل.
- از `sklearn.metrics` معیارهای ارزیابی (`accuracy_score`، `balanced_accuracy_score`، `precision_score`، `recall_score`، `f1_score`)، `classification_report`، `confusion_matrix` و `ConfusionMatrixDisplay` برای نمایش ماتریس درهم‌ریختگی.

</div>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

print("Libraries imported successfully.")

<div dir="rtl" style="text-align: right">

### بازسازی دقیق داده‌ی train و test
همان مراحل نوت‌بوک‌های قبلی تکرار می‌شود تا **دقیقاً همان** train و test و همان ستون‌ها به دست آید: بارگذاری، تقسیم (با `random_state=42`)، حذف ویژگی‌های تکراری و ثابت از روی train. این مهم است چون نام ویژگی‌های انتخاب‌شده باید در همین ستون‌ها وجود داشته باشد. `class_labels` فهرست مرتب‌شده‌ی کلاس‌هاست که در گزارش‌ها و ماتریس‌ها استفاده می‌شود (برای دیتاست دوکلاسه یا چندکلاسه مثل TCGA کار می‌کند).

</div>

In [ ]:
PROJECT_PATH = Path("/content/drive/MyDrive/WOA_Project")

import sys
sys.path.append(str(PROJECT_PATH))
from dataset_config import *   # <-- dataset-specific code lives only here

RESULTS_PATH = get_results_path(PROJECT_PATH)

df = load_dataset(PROJECT_PATH)

X, y, groups = split_xy(df)

X_train, X_test, y_train, y_test, groups_train = split_train_test(X, y, groups)

duplicate_columns = X_train.columns[
    X_train.T.duplicated()
].tolist()

X_train = X_train.drop(columns=duplicate_columns)
X_test = X_test.drop(columns=duplicate_columns)
X_train, X_test = drop_near_constant(X_train, X_test)

class_labels = sorted(y_train.unique())

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

<div dir="rtl" style="text-align: right">

### خواندن ویژگی‌های انتخاب‌شده‌ی WOA
فایل `final_woa_selected_features.csv` (خروجی نوت‌بوک ۰۵) از پوشه‌ی نتایج دیتاست فعال خوانده می‌شود و نام ویژگی‌ها به یک فهرست تبدیل می‌شود. تعداد و بیست نام اول چاپ می‌شود. اگر این فایل نباشد یعنی نوت‌بوک ۰۵ برای این دیتاست اجرا نشده است.

</div>

In [ ]:
selected_features_df = pd.read_csv(
    RESULTS_PATH / "final_woa_selected_features.csv"
)

selected_features = selected_features_df["Feature"].tolist()

print("Number of selected features:", len(selected_features))
print("First 20 selected features:")
print(selected_features[:20])

<div dir="rtl" style="text-align: right">

### اعتبارسنجی فهرست ویژگی‌ها
سه شرط بررسی می‌شود: فهرست خالی نباشد، ویژگی تکراری نداشته باشد، و همه‌ی ویژگی‌ها در ستون‌های `X_train` وجود داشته باشند. اگر شرطی برقرار نباشد، برنامه با خطا متوقف می‌شود تا با داده‌ی نامنطبق ادامه ندهیم (مثلاً اگر فایل نتیجه مال دیتاست دیگری باشد).

</div>

In [ ]:
assert len(selected_features) > 0
assert len(set(selected_features)) == len(selected_features)
assert set(selected_features).issubset(set(X_train.columns))

print("Selected features validation passed.")

<div dir="rtl" style="text-align: right">

### ساخت دو نسخه‌ی داده
- **Baseline:** همه‌ی ویژگی‌ها (`X_train` و `X_test` بدون تغییر).
- **WOA:** فقط ستون‌های انتخاب‌شده (`X_train[selected_features]`).

`.copy()` نسخه‌ی مستقل می‌سازد. ابعاد چاپ می‌شود؛ تعداد نمونه‌ها یکسان و تعداد ستون‌ها متفاوت است.

</div>

In [ ]:
X_train_baseline = X_train.copy()
X_test_baseline = X_test.copy()

X_train_woa = X_train[selected_features].copy()
X_test_woa = X_test[selected_features].copy()

print("Baseline training shape:", X_train_baseline.shape)
print("WOA training shape:", X_train_woa.shape)

print("Baseline testing shape:", X_test_baseline.shape)
print("WOA testing shape:", X_test_woa.shape)

<div dir="rtl" style="text-align: right">

### تعریف دو مدل یکسان
دو `Pipeline` با تنظیمات یکسان (استانداردسازی و Logistic Regression با `class_weight="balanced"`، `max_iter=5000`، `random_state=42`) ساخته می‌شود. چون مدل‌ها یکسان‌اند، هر تفاوت در نتیجه فقط از مجموعه‌ی ویژگی‌ها می‌آید.

</div>

In [ ]:
baseline_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=5000,
        random_state=42
    ))
])

woa_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=5000,
        random_state=42
    ))
])

print("Both models are ready.")

<div dir="rtl" style="text-align: right">

### آموزش مدل‌ها
هر مدل با `fit` روی train خودش آموزش می‌بیند: مدل پایه با همه‌ی ویژگی‌ها و مدل WOA با ویژگی‌های انتخاب‌شده. استانداردسازی هم در همین مرحله فقط از train یاد گرفته می‌شود.

</div>

In [ ]:
baseline_model.fit(X_train_baseline, y_train)

woa_model.fit(X_train_woa, y_train)

print("Both models trained successfully.")

<div dir="rtl" style="text-align: right">

### پیش‌بینی روی test
هر مدل برچسب نمونه‌های test را پیش‌بینی می‌کند. این‌جا اولین بار است که test توسط مدل دیده می‌شود. پیش‌بینی‌ها (`y_pred_baseline` و `y_pred_woa`) چاپ می‌شوند.

</div>

In [ ]:
y_pred_baseline = baseline_model.predict(X_test_baseline)

y_pred_woa = woa_model.predict(X_test_woa)

print("Predictions completed.")
print("Baseline predictions:", y_pred_baseline)
print("WOA predictions:", y_pred_woa)

<div dir="rtl" style="text-align: right">

### محاسبه‌ی معیارها و جدول مقایسه
تابع `evaluate_model` پنج معیار را حساب می‌کند:
- **Accuracy:** درصد پیش‌بینی‌های درست.
- **Balanced Accuracy:** میانگین recall هر کلاس.
- **Precision / Recall / F1 (Macro):** برای هر کلاس جدا حساب و با وزن برابر میانگین‌گیری می‌شود، پس کلاس کوچک هم به اندازه‌ی کلاس بزرگ اثر دارد.

نتیجه‌ی دو مدل در یک جدول (`comparison_df`) کنار هم قرار می‌گیرد و اسم سطرها تعداد ویژگی‌ها را نشان می‌دهد. این جدول اصلی‌ترین نتیجه‌ی پروژه است.

</div>

In [ ]:
def evaluate_model(y_true, y_pred):

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_true, y_pred),
        "Precision Macro": precision_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "Recall Macro": recall_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "F1 Macro": f1_score(
            y_true, y_pred, average="macro", zero_division=0
        )
    }


baseline_metrics = evaluate_model(
    y_test,
    y_pred_baseline
)

woa_metrics = evaluate_model(
    y_test,
    y_pred_woa
)

comparison_df = pd.DataFrame(
    [baseline_metrics, woa_metrics],
    index=[f"Baseline ({X_train.shape[1]} Features)", f"WOA ({len(selected_features)} Features)"]
)

comparison_df

<div dir="rtl" style="text-align: right">

### گزارش طبقه‌بندی مدل پایه
برای هر کلاس Precision (از پیش‌بینی‌های این کلاس چند درصد درست بود)، Recall (از نمونه‌های واقعی این کلاس چند درصد پیدا شد)، F1 و تعداد نمونه‌ی واقعی (support) نمایش داده می‌شود. برای دیدن این‌که مدل روی کدام کلاس ضعیف است مفید است.

</div>

In [ ]:
print("Baseline Classification Report")

print(
    classification_report(
        y_test,
        y_pred_baseline,
        labels=class_labels,
        zero_division=0
    )
)

<div dir="rtl" style="text-align: right">

### گزارش طبقه‌بندی مدل WOA
همان گزارش برای مدل با ویژگی‌های انتخاب‌شده. مقایسه‌ی آن با گزارش قبل نشان می‌دهد کاهش ویژگی‌ها روی کدام کلاس‌ها اثر مثبت یا منفی داشته است.

</div>

In [ ]:
print("WOA Classification Report")

print(
    classification_report(
        y_test,
        y_pred_woa,
        labels=class_labels,
        zero_division=0
    )
)

<div dir="rtl" style="text-align: right">

### ماتریس درهم‌ریختگی مدل پایه
ردیف‌ها کلاس واقعی و ستون‌ها کلاس پیش‌بینی‌شده‌اند. قطر اصلی پیش‌بینی‌های درست و خانه‌های بیرون قطر خطاها هستند؛ می‌بینیم کدام کلاس‌ها با هم اشتباه می‌شوند.

</div>

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_baseline,
    labels=class_labels,
    display_labels=[f"Class {c}" for c in class_labels],
    cmap="Blues",
    ax=ax
)

ax.set_title("Baseline Confusion Matrix")
plt.show()

<div dir="rtl" style="text-align: right">

### ماتریس درهم‌ریختگی مدل WOA
همان ماتریس برای مدل WOA. مقایسه‌ی دو ماتریس تعداد نمونه‌هایی را که با کاهش ویژگی‌ها درست یا غلط شده‌اند نشان می‌دهد.

</div>

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_woa,
    labels=class_labels,
    display_labels=[f"Class {c}" for c in class_labels],
    cmap="Blues",
    ax=ax
)

ax.set_title("WOA Confusion Matrix")
plt.show()

<div dir="rtl" style="text-align: right">

### نمودارهای مقایسه
دو نمودار کنار هم:
- **چپ:** مقایسه‌ی F1 Macro دو مدل (محور عمودی ۰ تا ۱).
- **راست:** تعداد ویژگی‌های هر مدل.

تصویری که از نمودارها معمولاً می‌خواهیم نشان دهیم: کیفیت تقریباً برابر با ویژگی‌های بسیار کمتر.

</div>

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Compare F1 Macro
comparison_df["F1 Macro"].plot(
    kind="bar",
    ax=axes[0],
    title="F1 Macro Comparison",
    ylim=(0, 1)
)

axes[0].set_ylabel("F1 Macro")
axes[0].tick_params(axis="x", rotation=15)

# Compare feature counts
axes[1].bar(
    ["Baseline", "WOA"],
    [X_train_baseline.shape[1], X_train_woa.shape[1]]
)

axes[1].set_title("Number of Selected Features")
axes[1].set_ylabel("Number of Features")

plt.tight_layout()
plt.show()

<div dir="rtl" style="text-align: right">

### درصد کاهش ویژگی‌ها
$$\text{reduction} = \frac{n_{all} - n_{selected}}{n_{all}} \times 100$$

تعداد ویژگی‌های اولیه، انتخاب‌شده و درصد کاهش چاپ می‌شود.

</div>

In [ ]:
original_feature_count = X_train_baseline.shape[1]
selected_feature_count = X_train_woa.shape[1]

reduction_percentage = (
    (original_feature_count - selected_feature_count)
    / original_feature_count
) * 100

print("Original features:", original_feature_count)
print("Selected features:", selected_feature_count)
print(f"Feature reduction: {reduction_percentage:.2f}%")

<div dir="rtl" style="text-align: right">

### ذخیره‌ی نتایج نهایی
در `results/<نام دیتاست>/` ذخیره می‌شود:
- `final_model_comparison.csv`: جدول مقایسه‌ی دو مدل،
- `baseline_classification_report.csv` و `woa_classification_report.csv`: گزارش کامل هر کلاس برای هر مدل (به‌صورت دیکشنری که به جدول تبدیل شده).

این فایل‌ها برای گزارش و ارائه استفاده می‌شوند.

</div>

In [ ]:
comparison_df.to_csv(
    RESULTS_PATH / "final_model_comparison.csv"
)

baseline_report = classification_report(
    y_test,
    y_pred_baseline,
    labels=class_labels,
    output_dict=True,
    zero_division=0
)

woa_report = classification_report(
    y_test,
    y_pred_woa,
    labels=class_labels,
    output_dict=True,
    zero_division=0
)

pd.DataFrame(baseline_report).to_csv(
    RESULTS_PATH / "baseline_classification_report.csv"
)

pd.DataFrame(woa_report).to_csv(
    RESULTS_PATH / "woa_classification_report.csv"
)

print("Final evaluation results saved successfully.")

In [ ]:
# جمع‌آوری تمام فایل‌های نتیجه‌ی دیتاست فعال در یک فایل PDF
from matplotlib.backends.backend_pdf import PdfPages

# توضیح کوتاه هر فایل نتیجه
result_descriptions = {
    "baseline_cv_results.csv": "نتایج اعتبارسنجی متقاطع مدل‌های پایه",
    "baseline_test_results.csv": "نتایج ارزیابی مدل‌های پایه روی داده‌ی test",
    "woa_selected_features.csv": "ویژگی‌های انتخاب‌شده توسط الگوریتم WOA",
    "woa_convergence.csv": "روند همگرایی الگوریتم WOA در طول تکرارها",
    "final_woa_selected_features.csv": "ویژگی‌های انتخاب‌شده در بهترین اجرای WOA",
    "woa_multiple_runs_summary.csv": "خلاصه‌ی نتایج اجراهای متعدد الگوریتم WOA",
    "final_model_comparison.csv": "مقایسه‌ی عملکرد مدل پایه و مدل دارای ویژگی‌های انتخاب‌شده‌ی WOA",
    "baseline_classification_report.csv": "گزارش طبقه‌بندی مدل پایه برای هر کلاس",
    "woa_classification_report.csv": "گزارش طبقه‌بندی مدل WOA برای هر کلاس"
}

result_files = sorted(RESULTS_PATH.glob("*.csv"))
pdf_path = RESULTS_PATH / f"{RESULTS_PATH.name}_all_results.pdf"

if not result_files:
    print("No result CSV files found in:", RESULTS_PATH)
else:
    with PdfPages(pdf_path) as pdf:
        for result_file in result_files:
            result_df = pd.read_csv(result_file)
            description = result_descriptions.get(
                result_file.name,
                "فایل نتایج ذخیره‌شده در پوشه‌ی نتایج"
            )

            # برای جدول‌های بزرگ، ردیف‌ها در چند صفحه قرار می‌گیرند.
            rows_per_page = 25
            chunks = [result_df.iloc[i:i + rows_per_page]
                      for i in range(0, max(len(result_df), 1), rows_per_page)]

            for page_number, chunk in enumerate(chunks, start=1):
                fig, ax = plt.subplots(figsize=(11.7, 8.3))
                ax.axis("off")
                ax.text(0.5, 0.97, f"Dataset: {RESULTS_PATH.name}",
                        ha="center", va="top", fontsize=16, fontweight="bold",
                        transform=ax.transAxes)
                ax.text(0.5, 0.91, description,
                        ha="center", va="top", fontsize=12,
                        transform=ax.transAxes)
                ax.text(0.5, 0.86, f"File: {result_file.name}",
                        ha="center", va="top", fontsize=9,
                        transform=ax.transAxes)

                if len(chunk.columns) > 0:
                    table = ax.table(
                        cellText=chunk.fillna("").astype(str).values,
                        colLabels=chunk.columns.astype(str),
                        cellLoc="center",
                        loc="center"
                    )
                    table.auto_set_font_size(False)
                    table.set_fontsize(7 if len(chunk.columns) > 8 else 9)
                    table.scale(1, 1.35)
                else:
                    ax.text(0.5, 0.5, "This file contains no columns.",
                            ha="center", va="center", transform=ax.transAxes)

                if len(chunks) > 1:
                    ax.text(0.98, 0.02, f"Page {page_number}/{len(chunks)}",
                            ha="right", va="bottom", fontsize=8,
                            transform=ax.transAxes)
                pdf.savefig(fig, bbox_inches="tight")
                plt.close(fig)

    print("Combined results PDF saved at:", pdf_path)
